<a href="https://colab.research.google.com/github/aavarela/SPBD_Labs/blob/main/projeto2/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Project 2
This work was carried out by Afonso Varela (73544) and Gonçalo Dionísio (73638), with the help of Google Gemini 3 Pro (AI agent).

#Environment configuration

In [ ]:
#@title Install & launch Kafka
%%bash
# 1. Kill any existing Kafka processes to free up port 9092
pkill -9 -f kafka || true

# 2. Clear out the temporary storage to avoid UUID mismatches
rm -rf /tmp/kafka-logs /tmp/kraft-combined-logs

# 3. Format and Start (Adding a check to ensure it actually stays up)
KAFKA_VERSION=3.7.2
KAFKA=kafka_2.12-$KAFKA_VERSION
UUID=`$KAFKA/bin/kafka-storage.sh random-uuid`
$KAFKA/bin/kafka-storage.sh format -t $UUID -c $KAFKA/config/server1.properties

# Start without -daemon for a moment to see immediate errors, OR check logs
$KAFKA/bin/kafka-server-start.sh -daemon $KAFKA/config/server1.properties

# 4. CRITICAL: Wait for the metadata quorum to be established
sleep 15

# 5. Verify the broker is actually listening
lsof -i :9092

In [ ]:
#@title Install pyspark 3.5.7 for compatibility with spark-sql-kafka 3.5.7
!pip uninstall -y dataproc-spark-connect
!pip uninstall -y pyspark
!pip install pyspark==3.5.7

In [ ]:
#@title Download 1% sample
!wget -q -O taxi_rides_1pc.csv.gz https://www.dropbox.com/scl/fi/v8ei5laqcalrx30z3lsty/taxi_rides_1pc.csv.gz?rlkey=q1lq7l56c4j97h9kymsdroau5&st=iurdwnwj&dl=0

In [ ]:
#@title Start Kafka publisher & Verify Stream (Fixed for Kafka 3.7.2)
import subprocess
import time
import os

# 1. Install dependencies
!pip --quiet install kafka-python dataclasses

# 2. Setup paths and files
KAFKA_PATH = "/content/kafka_2.12-3.7.2"
if not os.path.exists('kafka-publisher.py'):
    !wget -q -O kafka-publisher.py https://raw.githubusercontent.com/smduarte/spbd-2526/refs/heads/main/docs/labs/projs/kafka-publisher.py

# 3. Start publisher (Redirecting logs for debugging)
print("Starting Kafka publisher...")
!nohup python kafka-publisher.py --topic taxis_json --speedup 120 --filename taxi_rides_1pc.csv.gz > publisher.log 2>&1 &

# 4. Modern Health Check Loop (Replaces GetOffsetShell)
print("Verifying data stream (Waiting for offsets > 0)...")
data_detected = False

for i in range(20): # Try for ~100 seconds
    try:
        # Kafka 3.x tool to get the latest offset (--time -1 means 'latest')
        cmd = [
            f"{KAFKA_PATH}/bin/kafka-get-offsets.sh",
            "--bootstrap-server", "localhost:9092",
            "--topic", "taxis_json",
            "--time", "-1"
        ]

        output = subprocess.check_output(cmd, stderr=subprocess.STDOUT).decode()

        # Output format: taxis_json:0:OFFSETS
        if ":" in output:
            # Extract the offset (the last number after the last colon)
            current_offset = int(output.strip().split(':')[-1])

            if current_offset > 0:
                print(f"✅ Success! Kafka is streaming. Current offset: {current_offset}")
                data_detected = True
                break
    except Exception as e:
        # This handles cases where the topic isn't created yet or broker is warming up
        pass

    print(f"Waiting for data... ({i+1}/20)")
    time.sleep(5)

if not data_detected:
    print("❌ ERROR: No data detected. Showing last 10 lines of publisher.log:")
    !tail -n 10 publisher.log

# Data stream preparation

In [ ]:
#@title Collect data
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession \
    .builder \
    .appName('Kafka Spark Structured Streaming Example') \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7') \
    .getOrCreate()

lines = spark \
  .readStream \
  .format('kafka') \
  .option('kafka.bootstrap.servers', 'localhost:9092') \
  .option('subscribe', 'taxis_json') \
  .option('startingOffsets', 'earliest') \
  .load() \
  .selectExpr('CAST(value AS STRING)')

# Define a StructType named taxi_ride_schema
taxi_ride_schema = StructType([
    StructField("medallion", StringType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropoff_datetime", TimestampType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("pickup_grid_x", IntegerType(), True),
    StructField("pickup_grid_y", IntegerType(), True),
    StructField("dropoff_grid_x", IntegerType(), True),
    StructField("dropoff_grid_y", IntegerType(), True)
])

# Parsing and initial processing
parsed_stream = lines.withColumn("parsed_value", from_json(col("value"), taxi_ride_schema))
structured_stream = parsed_stream.select(col("parsed_value.*"))

MIN_LON = -74.916578
MAX_LAT = 41.47718278
LON_DELTA = 0.005986
LAT_DELTA = 0.004491556

processed_stream = structured_stream \
    .withColumn("pickup_grid_x", floor((MAX_LAT - col("pickup_latitude")) / LAT_DELTA)) \
    .withColumn("pickup_grid_y", floor((col("pickup_longitude") - MIN_LON) / LON_DELTA)) \
    .withColumn("dropoff_grid_x", floor((MAX_LAT - col("dropoff_latitude")) / LAT_DELTA)) \
    .withColumn("dropoff_grid_y", floor((col("dropoff_longitude") - MIN_LON) / LON_DELTA)) \
    .filter(
        (col("pickup_latitude").isNotNull()) & (col("pickup_longitude").isNotNull()) & \
        (col("dropoff_latitude").isNotNull()) & (col("dropoff_longitude").isNotNull()) & \
        (col("pickup_grid_x").between(0, 299)) & (col("pickup_grid_y").between(0, 299)) & \
        (col("dropoff_grid_x").between(0, 299)) & (col("dropoff_grid_y").between(0, 299)) & \
        (col("trip_distance") > 0)
    ) \
    .withColumn("trip_profit", col("fare_amount") + col("tip_amount")) \
    .withColumn("pickup_hour", hour(col("pickup_datetime"))) \
    .withColumn("time_of_day",
        when((col("pickup_hour") >= 0) & (col("pickup_hour") <= 5), "Night (0-5am)")
        .when((col("pickup_hour") >= 6) & (col("pickup_hour") <= 11), "Morning (6-11am)")
        .when((col("pickup_hour") >= 12) & (col("pickup_hour") <= 17), "Afternoon (12-5pm)")
        .otherwise("Evening (6-11pm)")
    )

# Watermarking
processed_stream_with_pickup_watermark = processed_stream.withWatermark("pickup_datetime", "1 minutes")
processed_stream_with_dropoff_watermark = processed_stream.withWatermark("dropoff_datetime", "1 minutes")

# Average profit per area stream
average_profit_per_area_stream = processed_stream_with_dropoff_watermark \
    .groupBy(
        window(col("dropoff_datetime"), "15 minutes"),
        col("pickup_grid_x"),
        col("pickup_grid_y"),
        col("time_of_day"),
    ) \
    .agg(
        avg("trip_profit").alias("average_trip_profit"),
        avg("trip_distance").alias("average_trip_distance"),
        count("*").alias("trip_count")
    )

# Empty taxis per area styream
dropoffs_for_empty = processed_stream_with_dropoff_watermark.select(
    col("medallion").alias("d_medallion"),
    col("dropoff_datetime").alias("d_time"),
    col("dropoff_grid_x").alias("d_grid_x"),
    col("dropoff_grid_y").alias("d_grid_y")
)

pickups_for_empty = processed_stream_with_pickup_watermark.select(
    col("medallion").alias("p_medallion"),
    col("pickup_datetime").alias("p_time"),
    col("pickup_grid_x").alias("p_grid_x"),
    col("pickup_grid_y").alias("p_grid_y")
)

empty_taxis_per_area_stream = dropoffs_for_empty.join(
    pickups_for_empty,
    expr("""
        d_medallion = p_medallion AND
        d_grid_x = p_grid_x AND
        d_grid_y = p_grid_y AND
        p_time BETWEEN d_time AND d_time + INTERVAL 30 MINUTES
    """),
    "leftOuter"
).filter(col("p_medallion").isNull()) \
 .groupBy(
    window(col("d_time"), "15 minutes"),
    col("d_grid_x"),
    col("d_grid_y")
).agg(count(col("d_medallion")).alias("empty_taxi_count"))

# Final joining and profitability calculation
area_profitability = average_profit_per_area_stream.join(
    empty_taxis_per_area_stream,
    (
        (average_profit_per_area_stream.window == empty_taxis_per_area_stream.window) &
        (average_profit_per_area_stream.pickup_grid_x == empty_taxis_per_area_stream.d_grid_x) &
        (average_profit_per_area_stream.pickup_grid_y == empty_taxis_per_area_stream.d_grid_y)
    ),
    "leftOuter"
).select(
    average_profit_per_area_stream.window,
    average_profit_per_area_stream.pickup_grid_x.alias("grid_x"),
    average_profit_per_area_stream.pickup_grid_y.alias("grid_y"),
    col("time_of_day"),
    col("average_trip_profit"),
    col("average_trip_distance"),
    col("trip_count"),
    coalesce(col("empty_taxi_count"), lit(0)).alias("empty_taxi_count")
).withColumn(
    "profitability",
    when(col("empty_taxi_count") == 0, 0.0)
    .otherwise(col("average_trip_profit") / col("empty_taxi_count"))
)

# Collect data from area profitability stream

In [ ]:
#@title OPTIONAL: test stream output
import time
import pandas as pd
from IPython.display import clear_output

# Stop any previous active queries
for q in spark.streams.active:
    q.stop()
print("All currently active streaming queries stopped.")

# Start a streaming query to collect the data
data_query = area_profitability \
    .writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("collected_data") \
    .start()

print("Streaming query started.")

try:
    print("Monitoring 'collected_data' table (Ctrl+C to stop early)...")
    while True:
        # Query the memory sink
        result = spark.sql("SELECT count(*) as total, count(profitability) as with_val FROM collected_data").toPandas()

        clear_output(wait=True)
        print(f"Last updated: {time.ctime()}")
        print(f"Total rows collected: {result['total'][0]}")
        print(f"Rows with profitability values: {result['with_val'][0]}")

        if result['total'][0] > 0:
            print("\nPreview:")
            display(spark.sql("SELECT * FROM collected_data LIMIT 5").toPandas())

        time.sleep(10)
except KeyboardInterrupt:
    print("Monitoring stopped.")

In [ ]:
import time
import pandas as pd

# Stop any previous active queries
for q in spark.streams.active:
    q.stop()
print("All currently active streaming queries stopped.")

# Start a streaming query to collect the data
data_query = area_profitability \
    .writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("collected_data") \
    .start()

print("Streaming query started.")

print("Waiting to allow data to be collected.")
time.sleep(7200)

# Stop the streaming query
data_query.stop()

print("Streaming query stopped. Data collection complete.")

# Retrieve the collected data from the memory sink into a Pandas DataFrame
area_profitability_df = spark.sql("SELECT * FROM collected_data").toPandas()

# Display the collected data
print("Displaying the first 5 rows of the collected data:")
display(area_profitability_df.head())

print(f"Total rows in area_profitability data: {len(area_profitability_df)}")

# 1. Analysis of the area profitability data

In [ ]:
#@title Visualize the data
import matplotlib.pyplot as plt
import pandas as pd

geo_df = area_profitability_df.groupby(['grid_x', 'grid_y']).agg({
    'profitability': ['mean', 'max']
}).reset_index()
geo_df.columns = ['grid_x', 'grid_y', 'avg_profit', 'max_profit']

plt.figure(figsize=(14, 10))

scatter = plt.scatter(
    geo_df['grid_x'],
    geo_df['grid_y'],
    c=geo_df['max_profit'],
    cmap='magma',
    s=120,
    marker='o',
    edgecolors='black',
    linewidth=0.3,
    alpha=0.8
)

plt.colorbar(scatter, label='Area Profitability')
plt.xlabel('Grid X')
plt.ylabel('Grid Y')
plt.title('NYC Taxi Area Profitability')
plt.grid(True, linestyle=':', alpha=0.4)

# Highlight the "Best" grid cell
best_row = geo_df.loc[geo_df['max_profit'].idxmax()]
plt.annotate(f"Highest Profitability Grid\n({best_row['grid_x']}, {best_row['grid_y']})",
              xy=(best_row['grid_x'], best_row['grid_y']),
              xytext=(best_row['grid_x']+10, best_row['grid_y']+5),
              arrowprops=dict(facecolor='white', shrink=0.05, width=1))

plt.show()

In [ ]:
#@title Calculate key summary statistics of the area profitability data
print("Key Summary Statistics for Average Profitability:")
display(aggregated_profitability['profitability'].describe())

In [ ]:
#@title Show histogram of the area profitability data
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram
plt.figure(figsize=(10, 6))
sns.histplot(area_profitability_df['profitability'], bins=50)
plt.title('Distribution of Area Profitability')
plt.xlabel('Profitability')
plt.ylabel('Frequency')
plt.show()


## Observations:
*   **Distribution Shape**: The histogram shows a skewed distribution, heavily concentrated towards lower profitability values, including a significant number of areas with zero profitability.
*   **Concentration at Zero**: A large peak at profitability = 0 indicates many grid areas likely had no trips or no successful trips during the observed time windows, or perhaps the formula resulted in zero for other reasons.
*   **Range**: While many areas have low profitability, there's a long tail extending to higher profitability values, suggesting a few highly profitable areas.
*   **Interpretation**: This implies that profitability is not uniformly distributed across the grid. There are specific areas that are very profitable, while a substantial number of areas yield little to no profit.

In [ ]:
#@title Show the box plot of the area profitability data
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x=area_profitability_df['profitability'])
plt.title('Box Plot of Area Profitability')
plt.xlabel('Profitability')
plt.show()

## Observations:
*   **Median and Quartiles**: The box plot clearly indicates that the median profitability is very low, likely at or near zero. The first quartile (Q1) and third quartile (Q3) are also low, reinforcing the observation from the histogram that most areas have low profitability.
*   **Outliers**: The box plot shows a considerable number of individual data points extending far above the upper whisker. These represent significant outliers – areas with exceptionally high profitability compared to the majority.
*   **Spread**: The interquartile range (IQR) is relatively small, indicating that the bulk of the data (the middle 50%) is tightly clustered at lower profitability values.
*   **Interpretation**: The box plot visually confirms the presence of highly profitable hotspots (outliers) that stand out from the vast majority of areas, which exhibit very low or zero profitability. This suggests that identifying and focusing on these high-profitability areas could be crucial for optimizing taxi operations.

## Analysis key findings
*   The profitability distribution is heavily skewed, with a significant concentration of areas showing low or zero profitability.
*   A prominent peak at profitability = 0 suggests many grid areas had no trips or no successful trips, or yield no profit.
*   While most areas have low profitability, there is a long tail extending to higher values, indicating the presence of a few exceptionally profitable areas.
*   The median profitability is very low, close to zero, and the first and third quartiles are also low, reinforcing that the majority of areas are not highly profitable.
*   The box plot clearly identifies a considerable number of outliers, representing areas with exceptionally high profitability, significantly above the median and the bulk of the data.
*   The interquartile range (IQR) is relatively small, showing that the central 50% of areas are clustered at very low profitability levels.

# 2. How does the trip profitability depend on the trip distance and time of day?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Setup the dashboard Layout
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
sns.set_theme(style="whitegrid")

# --- PLOT 1: Profitability vs distance ---
dist_stats = area_profitability_df.groupby('distance_bin', observed=False)['profitability'].mean().reset_index()

sns.barplot(
    data=dist_stats,
    x='distance_bin',
    y='profitability',
    hue='distance_bin',
    palette='viridis',
    legend=False,
    ax=ax1
)
ax1.set_title('Profitability vs. Trip Distance', fontsize=15, fontweight='bold')

# Add labels (values on top of bars)
for p in ax1.patches:
    if p.get_height() > 0:
        ax1.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 9), textcoords='offset points', fontweight='bold')

# --- PLOT 2: Profitability vs time of day ---
time_order = ["Night (0-5am)", "Morning (6-11am)", "Afternoon (12-5pm)", "Evening (6-11pm)"]
time_stats = area_profitability_df.groupby('time_of_day', observed=False)['profitability'].mean().reindex(time_order).reset_index()

sns.barplot(
    data=time_stats,
    x='time_of_day',
    y='profitability',
    hue='time_of_day',
    palette='magma',
    hue_order=time_order,
    legend=False,
    ax=ax2
)
ax2.set_title('Profitability vs. Time of Day', fontsize=15, fontweight='bold')

# Add labels (values on top of bars)
for p in ax2.patches:
    if p.get_height() > 0:
        ax2.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

# Addendum